# Tutorial 02 — Build and inspect a causal Transformer

[Lecture notes](../lectures/02_transformers.md) · [Exercise sheet](../exercises/02_transformers.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Implement a decoder forward pass in NumPy; inspect masked probabilities, vocabulary logits, and a future-token perturbation.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [ ]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


## 1. Attention with explicit shapes

This is one forward-pass decoder block with fixed random weights, not a pretrained model. The upper triangle must receive zero probability.

Before running: predict the output or the invariant being checked.


In [ ]:
def softmax(z):
    z=z-z.max(axis=-1,keepdims=True)
    e=np.exp(z)
    return e/e.sum(axis=-1,keepdims=True)
T,d,V = 6,8,12
emb=rng.normal(0,.2,(V,d)); pos=rng.normal(0,.2,(T,d))
Wq,Wk,Wv,Wo=[rng.normal(0,.2,(d,d)) for _ in range(4)]
W1=rng.normal(0,.2,(d,2*d)); W2=rng.normal(0,.2,(2*d,d))
head=rng.normal(0,.2,(d,V))
def norm(x):
    return (x-x.mean(-1,keepdims=True))/np.sqrt(x.var(-1,keepdims=True)+1e-5)
def decoder(tokens,causal=True):
    x=emb[tokens]+pos[:len(tokens)]
    z=norm(x); q,k,v=z@Wq,z@Wk,z@Wv
    scores=q@k.T/np.sqrt(d)
    if causal: scores=np.where(np.triu(np.ones(scores.shape,dtype=bool),1),-np.inf,scores)
    a=softmax(scores)
    h=x+(a@v)@Wo
    h=h+np.maximum(norm(h)@W1,0)@W2
    return norm(h)@head,a
seq=np.array([1,3,5,7,2,4])
logits,a=decoder(seq)
print("logits shape",logits.shape,"attention\n",np.round(a,3))
assert np.allclose(a.sum(-1),1)
assert np.allclose(a[np.triu_indices(T,1)],0)


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Test causality, not just plausible output

Change the suffix and verify all earlier outputs. Repeat without masking to expose the leak.

Before running: predict the output or the invariant being checked.


In [ ]:
changed=seq.copy(); changed[3:]=[9,8,10]
logits_changed,_=decoder(changed)
assert np.allclose(logits[:3],logits_changed[:3])
unmasked,_=decoder(seq,False); unmasked_changed,_=decoder(changed,False)
print("causal prefix difference",np.max(np.abs(logits[:3]-logits_changed[:3])))
print("unmasked prefix difference",np.max(np.abs(unmasked[:3]-unmasked_changed[:3])))
assert not np.allclose(unmasked[:3],unmasked_changed[:3])


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Shift the prediction task

Compute loss for each next-token target. The final observed token has no supplied next-token label in this sequence.

Before running: predict the output or the invariant being checked.


In [ ]:
pred=softmax(logits[:-1]); targets=seq[1:]
loss=-np.log(pred[np.arange(len(targets)),targets]).mean()
print("next-token loss",float(loss),"uniform reference",float(np.log(V)))
for cut in range(1,T):
    alt=seq.copy(); alt[cut:]=(alt[cut:]+1)%V
    assert np.allclose(decoder(seq)[0][:cut],decoder(alt)[0][:cut])


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why does masking after softmax without renormalization change the scale?
2. Can a decoder train all positions in parallel without seeing the future?
3. What property would an all-zero future mask test fail to detect?


## Extension and submission

Compare the block with a Qwen3 configuration using `extensions/hf_text.py`; identify head counts and normalization choices in the model configuration.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
